# Atividade 2 - Questao 2: Correcao Gama

Implementacao manual da transformacao gama em imagem monocromatica, sem uso de funcoes prontas de filtragem.

Formula usada:\n
\n
$B = 255 \cdot (A/255)^{(1/\gamma)}$

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

In [ ]:
BASE_DIR = Path.cwd()
if not (BASE_DIR / 'atividade2').exists():
    BASE_DIR = BASE_DIR.parent

ATV2_DIR = BASE_DIR / 'atividade2'
OUT_DIR = ATV2_DIR / 'saidas' / 'saidas_questao2'
OUT_DIR.mkdir(parents=True, exist_ok=True)

candidatos = [
    ATV2_DIR / 'imagem2.jpg',
    ATV2_DIR / 'image2.jpg',
    ATV2_DIR / 'image2.png',
]

INPUT_PATH = None
for p in candidatos:
    if p.exists():
        INPUT_PATH = p
        break

if INPUT_PATH is None:
    raise FileNotFoundError('Nao encontrei imagem2.jpg/image2.jpg/image2.png em atividade2/.')

print(f'Imagem de entrada: {INPUT_PATH}')
print(f'Pasta de saida: {OUT_DIR}')

In [ ]:
def load_grayscale_u8(path: Path) -> np.ndarray:
    img = Image.open(path).convert('L')
    return np.array(img, dtype=np.uint8)


def save_grayscale_u8(img_u8: np.ndarray, path: Path) -> None:
    Image.fromarray(img_u8).save(path)


def correcao_gama_manual(img_u8: np.ndarray, gamma: float) -> np.ndarray:
    if gamma <= 0:
        raise ValueError('gamma deve ser > 0')

    h, w = img_u8.shape
    out = np.zeros((h, w), dtype=np.uint8)
    inv_gamma = 1.0 / gamma

    for y in range(h):
        for x in range(w):
            r = float(img_u8[y, x]) / 255.0
            s = r ** inv_gamma
            val = int(round(s * 255.0))
            if val < 0:
                val = 0
            elif val > 255:
                val = 255
            out[y, x] = val

    return out

In [ ]:
imagem = load_grayscale_u8(INPUT_PATH)
gammas = [0.25, 0.5, 1.0, 1.5, 2.0, 3.0]

resultados = {}
metricas = {}

for gamma in gammas:
    img_g = correcao_gama_manual(imagem, gamma)
    resultados[gamma] = img_g

    out_name = f'imagem2_gama_{gamma}.png'
    out_path = OUT_DIR / out_name
    save_grayscale_u8(img_g, out_path)

    metricas[str(gamma)] = {
        'arquivo': str(out_path),
        'min': int(img_g.min()),
        'max': int(img_g.max()),
        'media': float(np.mean(img_g)),
        'desvio_padrao': float(np.std(img_g)),
        'shape': [int(img_g.shape[0]), int(img_g.shape[1])],
    }

print('Arquivos gerados:')
for g in gammas:
    print(f'- imagem2_gama_{g}.png')

In [ ]:
total = len(gammas) + 1
cols = 4
rows = int(np.ceil(total / cols))

plt.figure(figsize=(4 * cols, 3.5 * rows))

plt.subplot(rows, cols, 1)
plt.imshow(imagem, cmap='gray', vmin=0, vmax=255)
plt.title('Original')
plt.axis('off')

for i, gamma in enumerate(gammas, start=2):
    plt.subplot(rows, cols, i)
    plt.imshow(resultados[gamma], cmap='gray', vmin=0, vmax=255)
    plt.title(f'gamma = {gamma}')
    plt.axis('off')

plt.tight_layout()
comparativo_path = OUT_DIR / 'comparativo_gama.png'
plt.savefig(comparativo_path, dpi=150)
plt.show()
print(f'Comparativo salvo em: {comparativo_path}')

In [ ]:
payload = {
    'input': str(INPUT_PATH),
    'outdir': str(OUT_DIR),
    'gammas': gammas,
    'comparativo': str(OUT_DIR / 'comparativo_gama.png'),
    'resultados': metricas,
}

json_path = OUT_DIR / 'resultados_q2.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(payload, f, indent=2, ensure_ascii=False)

print(f'JSON salvo em: {json_path}')

In [ ]:
arquivos_esperados = [f'imagem2_gama_{g}.png' for g in gammas] + ['comparativo_gama.png', 'resultados_q2.json']

faltantes = []
for nome in arquivos_esperados:
    p = OUT_DIR / nome
    if not p.exists():
        faltantes.append(nome)

if faltantes:
    print('Faltando:', faltantes)
else:
    print('Verificacao OK: todos os arquivos foram gerados.')